In [1]:
import re
import unicodedata
from pathlib import Path

import pandas as pd

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_colwidth", 120)

In [2]:
arquivos = sorted(RAW_DIR.glob("*job_posts*.csv"))
vagas = pd.concat([pd.read_csv(f) for f in arquivos], ignore_index=True)
vagas = vagas.rename(columns={"keyword": "search_keyword"})
vagas = vagas[vagas["description_md"].notna()].reset_index(drop=True)

print(f"{len(arquivos)} arquivos lidos, {len(vagas)} vagas com descrição")
vagas["search_keyword"].value_counts()

1 arquivos lidos, 346 vagas com descrição


search_keyword
Engenheiro de IA                        89
LLM                                     54
Desenvolvedor de IA                     51
Engenheiro de Aprendizado de Máquina    49
IA Generativa                           28
Vibe Coding                             26
Engenharia de Prompt                    18
RAG                                     16
Agentes de IA                           15
Name: count, dtype: int64

In [3]:
vagas["url_base"] = vagas["url"].str.split("?").str[0]
CHAVE_CONTEUDO = ["job_title", "company", "description_md"]

keywords_por_vaga = (
    vagas.groupby(CHAVE_CONTEUDO)["search_keyword"]
    .agg(lambda s: sorted(set(s)))
    .rename("search_keywords")
)

antes = len(vagas)
vagas = (
    vagas.drop_duplicates(subset="url_base")
    .drop_duplicates(subset=CHAVE_CONTEUDO)
    .merge(keywords_por_vaga, on=CHAVE_CONTEUDO)
    .drop(columns="search_keyword")
)

print(f"{antes - len(vagas)} duplicadas removidas, {len(vagas)} vagas únicas")
print(f"{(vagas['search_keywords'].str.len() > 1).sum()} vagas encontradas por mais de um termo")

63 duplicadas removidas, 283 vagas únicas
6 vagas encontradas por mais de um termo


In [4]:
def limpar_texto(texto):
    texto = re.sub(r"<[^>]+>", " ", str(texto))
    texto = re.sub(r"\\([\-\.\+\#\(\)])", r"\1", texto)
    texto = re.sub(r"[ \t]+", " ", texto)
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    return texto.strip()


def normalizar(texto):
    texto = unicodedata.normalize("NFKD", texto.lower())
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r"[^a-z0-9\s\+\#\.]", " ", texto)
    texto = re.sub(r"\.(?![a-z0-9])", " ", texto)
    return re.sub(r"\s+", " ", texto).strip()


vagas["descricao"] = vagas["description_md"].map(limpar_texto)
vagas["descricao_norm"] = vagas["descricao"].map(normalizar)
vagas["n_caracteres"] = vagas["descricao"].str.len()

vagas["n_caracteres"].describe().round(0)

count      283.0
mean      3635.0
std       1858.0
min        116.0
25%       2526.0
50%       3209.0
75%       4590.0
max      11647.0
Name: n_caracteres, dtype: float64

In [5]:
from langdetect import DetectorFactory, detect

DetectorFactory.seed = 42


def detectar_idioma(texto):
    try:
        return detect(texto[:1500])
    except Exception:
        return "indeterminado"


vagas["idioma"] = vagas["descricao"].map(detectar_idioma)
vagas["idioma"].value_counts()

idioma
pt    167
en    116
Name: count, dtype: int64

In [6]:
SECOES = {
    "requisitos": r"requisit|qualifica|requirement|qualification|o que (esperamos|buscamos|procuramos)|looking for|what you( .ll)? need|you are a fit|perfil|conhecimento|skills|habilidade|diferenc|experi.ncia|forma..o acad|tecnologia|ferramenta",
    "responsabilidades": r"responsabilidade|atividade|atribui|responsibilit|what you( .ll| will)? (do|own)|you will be expected|como ser. (o )?seu dia|desafio|miss.o|role|job description|descri..o da vaga",
    "beneficios": r"benef.cio|benefit|o que oferecemos|what we offer|perk|remunera|oferecemos|what you get|make your work",
    "empresa": r"sobre (a|n.s)|about (us|the company)|company description|quem somos|nossa (hist.ria|cultura)",
}

CABECALHO_INLINE = re.compile(r"^\s*\*{2,3}(.{1,60}?)\*{2,3}")


def segmentar(texto):
    secao_atual = "outros"
    partes = {}
    for linha in texto.split("\n"):
        candidata = linha.strip().strip("*#:-• ").lower()
        if not 0 < len(candidata) < 60:
            inline = CABECALHO_INLINE.match(linha)
            candidata = inline.group(1).strip(" :").lower() if inline else ""
        if candidata:
            for nome, padrao in SECOES.items():
                if re.search(padrao, candidata):
                    secao_atual = nome
                    break
        partes.setdefault(secao_atual, []).append(linha)
    return {k: "\n".join(v).strip() for k, v in partes.items()}


segmentos = vagas["descricao"].map(segmentar)
for secao in list(SECOES) + ["outros"]:
    vagas[f"secao_{secao}"] = segmentos.map(lambda s, sec=secao: s.get(sec, ""))

cobertura = (vagas[[f"secao_{s}" for s in SECOES]] != "").mean().round(3)
print("Proporção de vagas com cada seção identificada:")
cobertura

Proporção de vagas com cada seção identificada:


secao_requisitos           0.894
secao_responsabilidades    0.618
secao_beneficios           0.530
secao_empresa              0.159
dtype: float64

In [7]:
amostra = vagas[vagas["secao_requisitos"] != ""].sample(3, random_state=42)
for _, vaga in amostra.iterrows():
    print(f"=== {vaga['job_title']} — {vaga['company']} ===")
    print(vaga["secao_requisitos"][:600])
    print()

=== Senior Machine Learning & LLM Engineer - Remote Work | REF#283560 — BairesDev ===
**What We Are Looking For*** 4+ years of experience in machine learning, AI, or related fields.
* Advanced expertise in Python and ML frameworks (e.g., TensorFlow, PyTorch, scikit-learn).
* Strong knowledge of NLP and LLM technologies.
* Proven track record in data preprocessing, feature engineering, and model optimization.
* Solid experience with code reviews, CI/CD processes, and scalable AI systems.
* Demonstrated ability to lead technical teams and communicate complex concepts effectively.
* Advanced level of English.

=== ESPECIALISTA EM INTELIGENCIA ARTIFICIAL — G4F ===
**Experiência*** Experiência em:Desenvolvimento de software
* Segurança da informação
* Operação e sustentação de ambientes de TIC
* Plataformas cloud, containers e automação

**Requisitos Técnicos*** Formação superior em TI ou graduação em qualquer área com pós-graduação em TI (mín. 360h – MEC)

Certificações (Pelo menos 2 certi

In [8]:
colunas = [
    "url_base", "search_keywords", "job_title", "company",
    "location", "salary", "experience", "job_type", "function", "industries",
    "descricao", "descricao_norm", "idioma", "n_caracteres",
    "secao_requisitos", "secao_responsabilidades", "secao_beneficios",
    "secao_empresa", "secao_outros",
]

vagas[colunas].to_parquet(PROCESSED_DIR / "vagas_limpas.parquet", index=False)
print(f"{len(vagas)} vagas salvas em {PROCESSED_DIR / 'vagas_limpas.parquet'}")

283 vagas salvas em ..\data\processed\vagas_limpas.parquet
